# H2o model

In [1]:
#import packages
import numpy as np
import pandas as pd 
#import matplotlib as mpl
import h2o
from h2o.automl import H2OAutoML
from h2o.estimators import H2OWord2vecEstimator, H2OGradientBoostingEstimator
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
h2o.init()

Checking whether there is an H2O instance running at http://localhost:54321 ..... not found.
Attempting to start a local H2O server...
  Java Version: openjdk version "11.0.10" 2021-01-19; OpenJDK Runtime Environment (build 11.0.10+9-Ubuntu-0ubuntu1.18.04); OpenJDK 64-Bit Server VM (build 11.0.10+9-Ubuntu-0ubuntu1.18.04, mixed mode, sharing)
  Starting server from /opt/conda/lib/python3.7/site-packages/h2o/backend/bin/h2o.jar
  Ice root: /tmp/tmpo3udptzm
  JVM stdout: /tmp/tmpo3udptzm/h2o_unknownUser_started_from_python.out
  JVM stderr: /tmp/tmpo3udptzm/h2o_unknownUser_started_from_python.err
  Server is running at http://127.0.0.1:54321
Connecting to H2O server at http://127.0.0.1:54321 ... successful.


H2O_cluster_uptime:,02 secs
H2O_cluster_timezone:,Etc/UTC
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.32.1.1
H2O_cluster_version_age:,1 month and 28 days
H2O_cluster_name:,H2O_from_python_unknownUser_4t7hh6
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,4 Gb
H2O_cluster_total_cores:,4
H2O_cluster_allowed_cores:,4
H2O_cluster_status:,"accepting new members, healthy"


In [3]:
job_titles = h2o.import_file('../input/commonlitreadabilityprize/train.csv')
test = h2o.import_file('../input/commonlitreadabilityprize/test.csv')
sample_submission = pd.read_csv('../input/commonlitreadabilityprize/sample_submission.csv')

Parse progress: |█████████████████████████████████████████████████████████| 100%
Parse progress: |█████████████████████████████████████████████████████████| 100%


In [4]:
print(job_titles.shape)
print(test.shape)

(2850, 6)
(8, 4)


In [5]:
#desscribe dataset
job_titles.head()

id,url_legal,license,excerpt,target,standard_error
c12129c31,,,"When the young people returned to the ballroom, it presented a decidedly changed appearance. Instead of an interior scene, it was a winter landscape. The floor was covered with snow-white canvas, not laid on smoothly, but rumpled over bumps and hillocks, like a real snow field. The numerous palms and evergreens that had decorated the room, were powdered with flour and strewn with tufts of cotton, like snow. Also diamond dust had been lightly sprinkled on them, and glittering crystal icicles hung from the branches. At each end of the room, on the wall, hung a beautiful bear-skin rug. These rugs were for prizes, one for the girls and one for the boys. And this was the game. The girls were gathered at one end of the room and the boys at the other, and one end was called the North Pole, and the other the South Pole. Each player was given a small flag which they were to plant on reaching the Pole. This would have been an easy matter, but each traveller was obliged to wear snowshoes.",-0.340259,0.464009
85aa80a4c,,,"All through dinner time, Mrs. Fayre was somewhat silent, her eyes resting on Dolly with a wistful, uncertain expression. She wanted to give the child the pleasure she craved, but she had hard work to bring herself to the point of overcoming her own objections. At last, however, when the meal was nearly over, she smiled at her little daughter, and said, ""All right, Dolly, you may go."" ""Oh, mother!"" Dolly cried, overwhelmed with sudden delight. ""Really? Oh, I am so glad! Are you sure you're willing?"" ""I've persuaded myself to be willing, against my will,"" returned Mrs. Fayre, whimsically. ""I confess I just hate to have you go, but I can't bear to deprive you of the pleasure trip. And, as you say, it would also keep Dotty at home, and so, altogether, I think I shall have to give in."" ""Oh, you angel mother! You blessed lady! How good you are!"" And Dolly flew around the table and gave her mother a hug that nearly suffocated her.",-0.315372,0.480805
b69ac6792,,,"As Roger had predicted, the snow departed as quickly as it came, and two days after their sleigh ride there was scarcely a vestige of white on the ground. Tennis was again possible and a great game was in progress on the court at Pine Laurel. Patty and Roger were playing against Elise and Sam Blaney, and the pairs were well matched. But the long-contested victory finally went against Patty, and she laughingly accepted defeat. ""Only because Patty's not quite back on her game yet,"" Roger defended; ""this child has been on the sick list, you know, Sam, and she isn't up to her own mark."" ""Well, I like that!"" cried Patty; ""suppose you bear half the blame, Roger. You see, Mr. Blaney, he is so absorbed in his own Love Game, he can't play with his old-time skill."" ""All right, Patsy, let it go at that. And it's so, too. I suddenly remembered something Mona told me to tell you, and it affected my service.""",-0.580118,0.476676
dd1000b26,,,"And outside before the palace a great garden was walled round, filled full of stately fruit-trees, gray olives and sweet figs, and pomegranates, pears, and apples, which bore the whole year round. For the rich south-west wind fed them, till pear grew ripe on pear, fig on fig, and grape on grape, all the winter and the spring. And at the farther end gay flower-beds bloomed through all seasons of the year; and two fair fountains rose, and ran, one through the garden grounds, and one beneath the palace gate, to water all the town. Such noble gifts the heavens had given to Alcinous the wise. So they went in, and saw him sitting, like Poseidon, on his throne, with his golden sceptre by him, in garments stiff with gold, and in his hand a sculptured goblet, as he pledged the merchant kings; and beside him stood Arete, his wise and lovely queen, and leaned against a pillar as she spun her golden threads.",-1.05401,0.450007
37c1b32fb,,,"Once upon a time there were Three

In [6]:
STOP_WORDS = ["ax","i","you","edu","s","t","m","subject","can",
              "lines","re","what","there","all","we","one","the",
              "a","an","of","or","in","for","by","on","but","is",
              "in","a","not","with","as","was","if","they","are",
              "this","and","it","have","from","at","my","be","by",
              "not","that","to","from","com","org","like","likes",
              "so"]

In [7]:
def tokenize(sentences, stop_word = STOP_WORDS):
    tokenized = sentences.tokenize("\\W+")
    tokenized_lower = tokenized.tolower()
    tokenized_filtered = tokenized_lower[(tokenized_lower.nchar() >= 2) | (tokenized_lower.isna()),:]
    tokenized_words = tokenized_filtered[tokenized_filtered.grep("[0-9]",invert=True,output_logical=True),:]
    tokenized_words = tokenized_words[(tokenized_words.isna()) | (~ tokenized_words.isin(STOP_WORDS)),:]
    return tokenized_words

In [8]:
def predict(job_title,w2v, gbm):
    words = tokenize(h2o.H2OFrame(job_title).ascharacter())
    job_title_vec = w2v.transform(words, aggregate_method="AVERAGE")
    print(gbm.predict(test_data=job_title_vec))
    return (gbm.predict(test_data=job_title_vec))

In [9]:
words = tokenize(job_titles["excerpt"])

In [10]:
w2v_model = H2OWord2vecEstimator(sent_sample_rate = 0.0, epochs = 2)
w2v_model.train(training_frame=words)

word2vec Model Build progress: |██████████████████████████████████████████| 100%


In [11]:
w2v_model.find_synonyms("teacher", count = 5)

OrderedDict([('neighbors', 0.8936282992362976),
             ('okay', 0.8931559324264526),
             ('hurt', 0.8908748030662537),
             ('gladly', 0.8906795978546143),
             ('majesty', 0.890442967414856)])

In [12]:
# Calculate a vector for each job title:
job_title_vecs = w2v_model.transform(words, aggregate_method = "AVERAGE")

In [13]:
# Prepare training & validation data (keep only job titles made of known words):
valid_job_titles = ~ job_title_vecs["C4"].isna()
data = job_titles[valid_job_titles,:].cbind(job_title_vecs[valid_job_titles,:])
data_split = data.split_frame(ratios=[0.8])

In [14]:
# Build a basic GBM model:
gbm_model = H2OGradientBoostingEstimator()
gbm_model.train(x = job_title_vecs.names,
                y="target",
                training_frame = data_split[0],
                validation_frame = data_split[1])

gbm Model Build progress: |███████████████████████████████████████████████| 100%


In [15]:
perf = gbm_model.model_performance()
perf


ModelMetricsRegression: gbm
** Reported on train data. **

MSE: 0.25712624452456484
RMSE: 0.507076172310004
MAE: 0.39722341822336965
RMSLE: NaN
Mean Residual Deviance: 0.25712624452456484


In [16]:
test = pd.read_csv('../input/commonlitreadabilityprize/test.csv')
test["target"] = float(1)
test1=np.zeros(7)

In [17]:
# Predict
for i in range(0,7):
    print(test["target"][i])
    a=predict([test["excerpt"][i]],w2v_model, gbm_model)
    test["target"][i]=a["predict"]
    print(test["target"][i])
#print(predict(["school teacher having holidays every month"], w2v_model, gbm_model))
a

1.0
Parse progress: |█████████████████████████████████████████████████████████| 100%
gbm prediction progress: |████████████████████████████████████████████████| 100%


predict
-0.956903



gbm prediction progress: |████████████████████████████████████████████████| 100%


/opt/conda/lib/python3.7/site-packages/ipykernel_launcher.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  """


-0.9569030610933981
1.0
Parse progress: |█████████████████████████████████████████████████████████| 100%
gbm prediction progress: |████████████████████████████████████████████████| 100%


...

predict
-0.650044



gbm prediction progress: |████████████████████████████████████████████████| 100%
-0.6500437159835558
1.0
Parse progress: |█████████████████████████████████████████████████████████| 100%
gbm prediction progress: |████████████████████████████████████████████████| 100%


predict
0.203529



gbm prediction progress: |████████████████████████████████████████████████| 100%
0.20352886172018836


predict
0.203529


In [18]:
test["target"]

0   -0.956903
1   -0.578524
2   -0.538965
3   -1.232306
4   -1.363716
5   -0.650044
6    0.203529
Name: target, dtype: float64

In [19]:
sample_submission["target"]=test["target"]
sample_submission.head()

,id,target
0,c0f722661,-0.956903
1,f0953f0a5,-0.578524
2,0df072751,-0.538965
3,04caf4e0c,-1.232306
4,0e63f8bea,-1.363716


In [20]:
sample_submission.to_csv('submission.csv', index=False)